## Feature Extraction using CNN of Public Nusaince Analysis 


In [ ]:
!pip install torch torchvision

In [ ]:
import cv2
import torch
import matplotlib.pyplot as plt

from torchvision import models, transforms
from PIL import Image

In [ ]:
model = models.resnet18(weights="DEFAULT")

model.eval()

print("ResNet18 loaded")

In [ ]:
feature_extractor = torch.nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
video_path = "../data/raw/Abuse/sample.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

cap.release()

frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.imshow(frame)
plt.title("Input Frame")
plt.axis("off")
plt.show()

In [ ]:
image = Image.fromarray(frame)

input_tensor = transform(image)

input_tensor = input_tensor.unsqueeze(0)

print("Input shape:", input_tensor.shape)

In [ ]:
with torch.no_grad():

    features = feature_extractor(input_tensor)

print("Feature shape:", features.shape)

In [ ]:
features = features.view(features.size(0), -1)

print("Feature vector shape:", features.shape)

In [ ]:
def extract_features(frames):

    feature_vectors = []

    for frame in frames:

        image = Image.fromarray(frame)

        tensor = transform(image)
        tensor = tensor.unsqueeze(0)

        with torch.no_grad():
            feature = feature_extractor(tensor)

        feature = feature.view(1, -1)

        feature_vectors.append(
            feature.squeeze(0)
        )

    return torch.stack(feature_vectors)

In [ ]:
features = extract_features(frames[:16])

print("Feature shape:", features.shape)